<a href="https://colab.research.google.com/github/zeets13/Flaggr_Project/blob/main/Baseline_Multi_Label.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset
dataset = load_dataset(
    "ucberkeley-dlab/measuring-hate-speech",
    "default"
)
df = dataset["train"].to_pandas()
print("Rows:", len(df))
print("Columns:", len(df.columns))


README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/135556 [00:00<?, ? examples/s]

Rows: 135556
Columns: 143


In [ ]:
category_columns = [
    "insult",
    "humiliate",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend"
]

print(df[category_columns].head())

   insult  humiliate  dehumanize  violence  genocide  attack_defend
0     0.0        0.0         0.0       0.0       0.0            0.0
1     0.0        0.0         0.0       0.0       0.0            2.0
2     4.0        4.0         4.0       0.0       0.0            4.0
3     2.0        1.0         0.0       0.0       0.0            3.0
4     4.0        4.0         4.0       4.0       1.0            3.0


In [ ]:
agg_dict = {
    "text": "first",
    "hatespeech": list
}

for col in category_columns:
    agg_dict[col] = "mean"

df_grouped = (
    df.groupby("comment_id")
      .agg(agg_dict)
      .reset_index()
)

print(df_grouped.shape)

(39565, 9)


In [ ]:
for col in category_columns:
    df_grouped[col + "_label"] = (df_grouped[col] >= 2).astype(int)

multi_label_columns = [
    "insult_label",
    "humiliate_label",
    "dehumanize_label",
    "violence_label",
    "genocide_label",
    "attack_defend_label"
]
for col in multi_label_columns:

    print("\n", col)
    print(df_grouped[col].value_counts())



 insult_label
insult_label
1    28934
0    10631
Name: count, dtype: int64

 humiliate_label
humiliate_label
1    26053
0    13512
Name: count, dtype: int64

 dehumanize_label
dehumanize_label
0    20884
1    18681
Name: count, dtype: int64

 violence_label
violence_label
0    33410
1     6155
Name: count, dtype: int64

 genocide_label
genocide_label
0    36834
1     2731
Name: count, dtype: int64

 attack_defend_label
attack_defend_label
1    32644
0     6921
Name: count, dtype: int64


In [ ]:
def majority_label(values):
    return pd.Series(values).mode()[0]
df_grouped["majority_hatespeech"] = (
    df_grouped["hatespeech"]
    .apply(majority_label)
)
print(
    df_grouped["majority_hatespeech"]
    .value_counts()
    .sort_index()
)

df_grouped["binary_label"] = (
    df_grouped["majority_hatespeech"] == 2
).astype(int)
print(
    df_grouped["binary_label"]
    .value_counts()
    .sort_index()
)

df_multi = df_grouped[
    [
        "comment_id",
        "text",
        "binary_label"
    ] + multi_label_columns
].copy()

print(df_multi.shape)
print(df_multi.columns.tolist())

majority_hatespeech
0.0    29304
1.0     1933
2.0     8328
Name: count, dtype: int64
binary_label
0    31237
1     8328
Name: count, dtype: int64
(39565, 9)
['comment_id', 'text', 'binary_label', 'insult_label', 'humiliate_label', 'dehumanize_label', 'violence_label', 'genocide_label', 'attack_defend_label']


In [ ]:

stratify_columns = [
    "binary_label",
    "insult_label",
    "humiliate_label",
    "dehumanize_label",
    "violence_label",
    "genocide_label",
    "attack_defend_label"
]

for col in stratify_columns:

    print(
        f"{col}: "
        f"{df_multi[col].mean():.4f}"
    )
!pip install iterative-stratification
from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

Y = df_multi[stratify_columns].values
X = df_multi.index.values.reshape(-1, 1)

target_size = 15000

remove_fraction = (
    1 - target_size / len(df_multi)
)

print("Remove fraction:", remove_fraction)
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=remove_fraction,
    random_state=42
)

selected_idx, _ = next(
    msss.split(X, Y)
)
df_15000 = df_multi.iloc[
    selected_idx
].copy()

df_15000 = df_15000.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)
print("Dataset size:", len(df_15000))

binary_label: 0.2105
insult_label: 0.7313
humiliate_label: 0.6585
dehumanize_label: 0.4722
violence_label: 0.1556
genocide_label: 0.0690
attack_defend_label: 0.8251
Remove fraction: 0.6208770377859218
Dataset size: 15000


In [ ]:
print(" ORIGINAL DATASET")

for col in stratify_columns:

    print(
        f"{col}: "
        f"{df_multi[col].mean():.6f}"
    )


print("\nSample Dataset")

for col in stratify_columns:

    print(
        f"{col}: "
        f"{df_15000[col].mean():.6f}"
    )

Y_15000 = df_15000[
    stratify_columns
].values

X_15000 = df_15000.index.values.reshape(-1, 1)
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    msss.split(X_15000, Y_15000)
)
df_train = df_15000.iloc[
    train_idx
].copy()

df_temp = df_15000.iloc[
    temp_idx
].copy()
print("Train:", len(df_train))
print("Temporary:", len(df_temp))


Y_temp = df_temp[
    stratify_columns
].values

X_temp = df_temp.index.values.reshape(-1, 1)
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    msss.split(X_temp, Y_temp)
)
df_valid = df_temp.iloc[
    val_idx
].copy()

df_test = df_temp.iloc[
    test_idx
].copy()

print("Train:", len(df_train))
print("Validation:", len(df_valid))
print("Test:", len(df_test))

 ORIGINAL DATASET
binary_label: 0.210489
insult_label: 0.731303
humiliate_label: 0.658486
dehumanize_label: 0.472160
violence_label: 0.155567
genocide_label: 0.069026
attack_defend_label: 0.825073

Sample Dataset
binary_label: 0.210467
insult_label: 0.731333
humiliate_label: 0.658467
dehumanize_label: 0.472133
violence_label: 0.155600
genocide_label: 0.069000
attack_defend_label: 0.825067
Train: 12000
Temporary: 3000
Train: 12000
Validation: 1500
Test: 1500


In [ ]:
df_train.shape

(12000, 9)

In [ ]:
for name, dataset in [
    ("TRAIN", df_train),
    ("VALIDATION", df_valid),
    ("TEST", df_test)
]:

    print(name)

    for col in stratify_columns:

        print(
            f"{col}: "
            f"{dataset[col].mean():.4f}"
        )

TRAIN
binary_label: 0.2105
insult_label: 0.7313
humiliate_label: 0.6585
dehumanize_label: 0.4722
violence_label: 0.1556
genocide_label: 0.0690
attack_defend_label: 0.8251
VALIDATION
binary_label: 0.2100
insult_label: 0.7313
humiliate_label: 0.6580
dehumanize_label: 0.4720
violence_label: 0.1560
genocide_label: 0.0687
attack_defend_label: 0.8253
TEST
binary_label: 0.2107
insult_label: 0.7313
humiliate_label: 0.6587
dehumanize_label: 0.4720
violence_label: 0.1553
genocide_label: 0.0693
attack_defend_label: 0.8247


In [ ]:
from datasets import Dataset
model_columns = [
    "text"
] + multi_label_columns

train_dataset = Dataset.from_pandas(
    df_train[model_columns],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    df_valid[model_columns],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    df_test[model_columns],
    preserve_index=False
)

def create_labels(example):

    example["labels"] = [
        float(example[col])
        for col in multi_label_columns
    ]

    return example
train_dataset = train_dataset.map(
    create_labels
)

valid_dataset = valid_dataset.map(
    create_labels
)

test_dataset = test_dataset.map(
    create_labels
)

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
df_train.to_csv('/content/drive/MyDrive/Untitled Folder/train.csv', index=False)
df_test.to_csv('/content/drive/MyDrive/Untitled Folder/test.csv', index=False)
df_valid.to_csv('/content/drive/MyDrive/Untitled Folder/valid.csv', index=False)


In [ ]:
df_train.head()

,comment_id,text,binary_label,insult_label,humiliate_label,dehumanize_label,violence_label,genocide_label,attack_defend_label
0,38566,"Stick It, Fuck It, Breed It, Leave It #barebac...",0,1,1,1,0,0,0
1,21652,"I will never be fucked up bout no nigga again,...",0,1,0,0,0,0,1
3,3010,"Gallagher killed a brown person so who cares, ...",0,1,1,1,1,1,1
4,20088,I hate niggas you complain like suck it up you...,1,1,1,0,0,0,1
5,22471,stupid BITCH,1,1,1,1,0,0,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = df_train["text"].fillna("").astype(str)
X_val = df_valid["text"].fillna("").astype(str)
X_test = df_test["text"].fillna("").astype(str)

y_train = df_train[multi_label_columns].astype(int)
y_val = df_valid[multi_label_columns].astype(int)
y_test = df_test[multi_label_columns].astype(int)

print("X train:", X_train.shape)
print("y train:", y_train.shape)

X train: (12000,)
y train: (12000, 6)


In [ ]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (12000, 30000)
Validation TF-IDF shape: (1500, 30000)
Test TF-IDF shape: (1500, 30000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

lr_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
)

lr_model.fit(
    X_train_tfidf,
    y_train
)

print("Training completed.")

Training completed.


In [ ]:
y_pred = lr_model.predict(X_test_tfidf)
y_prob = lr_model.predict_proba(X_test_tfidf)

print("Prediction shape:", y_pred.shape)
print("Probability shape:", y_prob.shape)

Prediction shape: (1500, 6)
Probability shape: (1500, 6)


In [ ]:
from sklearn.metrics import classification_report

for i, label in enumerate(multi_label_columns):

    print(label)


    print(
        classification_report(
            y_test.iloc[:, i],
            y_pred[:, i],
            target_names=["Negative", "Positive"],
            zero_division=0
        )
    )

insult_label
              precision    recall  f1-score   support

    Negative       0.58      0.73      0.64       403
    Positive       0.89      0.80      0.84      1097

    accuracy                           0.78      1500
   macro avg       0.73      0.77      0.74      1500
weighted avg       0.81      0.78      0.79      1500

humiliate_label
              precision    recall  f1-score   support

    Negative       0.60      0.71      0.65       512
    Positive       0.83      0.75      0.79       988

    accuracy                           0.74      1500
   macro avg       0.71      0.73      0.72      1500
weighted avg       0.75      0.74      0.74      1500

dehumanize_label
              precision    recall  f1-score   support

    Negative       0.71      0.71      0.71       792
    Positive       0.67      0.68      0.67       708

    accuracy                           0.69      1500
   macro avg       0.69      0.69      0.69      1500
weighted avg       0.69     

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

subset_accuracy = accuracy_score(y_test, y_pred)

micro_precision = precision_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob,
    average="macro"
)


print(f"Subset Accuracy: {subset_accuracy:.4f}")
print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall: {micro_recall:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")
print(f"Macro ROC-AUC: {roc_auc:.4f}")

Subset Accuracy: 0.3320
Micro Precision: 0.7905
Micro Recall: 0.7559
Micro F1: 0.7728
Macro F1 : 0.6749
Macro ROC-AUC: 0.7978


In [ ]:
y_val_pred = lr_model.predict(X_val_tfidf)
y_val_prob = lr_model.predict_proba(X_val_tfidf)

print(
    "Micro F1:",
    f1_score(
        y_val,
        y_val_pred,
        average="micro",
        zero_division=0
    )
)

print(
    "Macro F1:",
    f1_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val,
        y_val_prob,
        average="macro"
    )
)

Micro F1: 0.776056338028169
Macro F1: 0.6816510668066265
ROC-AUC: 0.8146111789738647


In [ ]:
import joblib

joblib.dump(tfidf, "/content/drive/MyDrive/Untitled Folder/tfidf_multilabel.pkl")
joblib.dump(lr_model, "/content/drive/MyDrive/Untitled Folder/lr_multilabel.pkl")

print("TF-IDF vectorizer and LR model saved.")

TF-IDF vectorizer and LR model saved.
